# Normalise relation predicates

Some predicates appear twice in `ALL_relations.json`, once with spaces and once with underscores (for example `has unit` and `has_unit`). This notebook goes through every relation, replaces the spaces in its `relation` field with underscores, and saves the file. It then rebuilds `data.js` so the viewer shows the merged predicates.

Run the cells in order from the `kg_viewer2` folder. Running it again is safe: once every predicate uses underscores, nothing changes.

In [1]:
import json
import re
import shutil
import subprocess
import sys
from collections import Counter
from pathlib import Path

DATA_FILE = Path('ALL_relations.json')
BACKUP_FILE = Path('ALL_relations.backup.json')

with DATA_FILE.open(encoding='utf-8-sig') as stream:
    relations = json.load(stream)

print(f'Loaded {len(relations)} relations')
print(f'Unique predicates before: {len({r["relation"] for r in relations})}')

Loaded 2077 relations
Unique predicates before: 757


In [2]:
def normalize(predicate):
    # Trim the ends, then turn each run of spaces into one underscore.
    return re.sub(r'\s+', '_', predicate.strip())

changes = Counter()
for relation in relations:
    old = relation['relation']
    new = normalize(old)
    if new != old:
        changes[(old, new)] += 1
        relation['relation'] = new

print(f'Rows changed: {sum(changes.values())}')
print(f'Distinct predicates rewritten: {len(changes)}')
for (old, new), count in sorted(changes.items()):
    print(f'  {old!r} -> {new!r}  ({count} rows)')

Rows changed: 290
Distinct predicates rewritten: 173
  'absent in' -> 'absent_in'  (1 rows)
  'accounts for' -> 'accounts_for'  (1 rows)
  'allows analysis of' -> 'allows_analysis_of'  (1 rows)
  'allows to draw conclusions on' -> 'allows_to_draw_conclusions_on'  (1 rows)
  'applied according to' -> 'applied_according_to'  (1 rows)
  'applied at' -> 'applied_at'  (2 rows)
  'applied by' -> 'applied_by'  (1 rows)
  'applied to' -> 'applied_to'  (2 rows)
  'applied with' -> 'applied_with'  (1 rows)
  'applies to' -> 'applies_to'  (1 rows)
  'are reaction-dominated' -> 'are_reaction-dominated'  (1 rows)
  'assumed to be' -> 'assumed_to_be'  (1 rows)
  'behaves as' -> 'behaves_as'  (1 rows)
  'calculated as' -> 'calculated_as'  (1 rows)
  'calculated by subtracting' -> 'calculated_by_subtracting'  (2 rows)
  'calculated from' -> 'calculated_from'  (1 rows)
  'calculated using' -> 'calculated_using'  (2 rows)
  'cause significant change in' -> 'cause_significant_change_in'  (1 rows)
  'caus

In [3]:
unique_after = sorted({r['relation'] for r in relations})
assert not any(' ' in p for p in unique_after), 'A predicate still contains a space'
print(f'Unique predicates after: {len(unique_after)}')

Unique predicates after: 692


In [4]:
# Keep a copy of the original file the first time this runs.
if not BACKUP_FILE.exists():
    shutil.copyfile(DATA_FILE, BACKUP_FILE)
    print(f'Backup written to {BACKUP_FILE}')

# Same formatting as the original file, so only the relation lines change.
with DATA_FILE.open('w', encoding='utf-8', newline='\n') as stream:
    json.dump(relations, stream, ensure_ascii=False, indent=2)
    stream.write('\n')
print(f'Saved {DATA_FILE}')

Backup written to ALL_relations.backup.json
Saved ALL_relations.json


In [5]:
# Rebuild data.js so the viewer picks up the merged predicates.
result = subprocess.run([sys.executable, 'build_data.py'], capture_output=True, text=True)
print(result.stdout or result.stderr)
result.check_returncode()

Bundled 2077 relations from C:\Users\jaina\OneDrive\Desktop\kg_viewer2\ALL_relations.json into C:\Users\jaina\OneDrive\Desktop\kg_viewer2\data.js

